In [ ]:
from google.colab import drive
import os
import json
from datetime import datetime

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Constantes de ruta y configuración
DIRECTORIO = '/content/drive/My Drive/Lenguaje II'
EXTENSIONES_VALIDAS = ('.json', '.csv')
ARCHIVO_INVENTARIO = 'inventario.json'


# 3. Función modular estructurada con manejo de excepciones
def list_files_in_directory(directorio):
    resultado = {
        'directorio': directorio,
        'estado': 'ok',
        'mensaje': '',
        'total_archivos': 0,
        'archivos': [],
        'inventario': {'existe': False, 'valido': False,
                       'total_registros': 0, 'registros': [], 'error': None},
    }

    try:
        archivos = os.listdir(directorio)
    except FileNotFoundError:
        print('Directorio no encontrado.')
        resultado['estado'] = 'error'
        resultado['mensaje'] = 'Directorio no encontrado.'
        return resultado

    print(f'Archivos en {directorio}:')
    for a in sorted(archivos):
        ruta = os.path.join(directorio, a)

        # filtrado selectivo (solo .json o .csv)
        if not os.path.isfile(ruta) or not a.lower().endswith(EXTENSIONES_VALIDAS):
            continue

        # metadatos (tamaño en KB y fecha de última modificación)
        tamano_kb = round(os.path.getsize(ruta) / 1024, 2)
        fecha = datetime.fromtimestamp(os.path.getmtime(ruta)).strftime('%Y-%m-%d %H:%M:%S')

        print(f' └── {a} | {tamano_kb} KB | modificado: {fecha}')
        resultado['archivos'].append({
            'nombre': a,
            'extension': os.path.splitext(a)[1].lower(),
            'tamano_kb': tamano_kb,
            'ultima_modificacion': fecha,
        })

    resultado['total_archivos'] = len(resultado['archivos'])
    resultado['mensaje'] = f"{resultado['total_archivos']} archivo(s) encontrados."

    # parseo estructurado de inventario.json
    ruta_inv = os.path.join(directorio, ARCHIVO_INVENTARIO)
    if os.path.isfile(ruta_inv):
        inv = resultado['inventario']
        inv['existe'] = True
        try:
            with open(ruta_inv, 'r', encoding='utf-8') as f:
                datos = json.load(f)
            if isinstance(datos, list) and all(isinstance(r, dict) for r in datos):
                inv.update(valido=True, total_registros=len(datos), registros=datos)
                print(f'\n{ARCHIVO_INVENTARIO} válido: {len(datos)} registros.')
            else:
                inv['error'] = 'El JSON no es una lista de registros válida.'
                print(f'\n{ARCHIVO_INVENTARIO} inválido: {inv["error"]}')
        except json.JSONDecodeError as e:
            inv['error'] = f'JSON mal formado: {e}'
            print(f'\n{ARCHIVO_INVENTARIO} inválido: {inv["error"]}')

    # retorno modular (diccionario listo para una API)
    return resultado


if __name__ == '__main__':
    resultado = list_files_in_directory(DIRECTORIO)